# Lambda Baseline Comparison

Tests whether the KDE lambda model (`critic_model.py`) outperforms naive baselines,
and whether blending the KDE prior with observed rolling rate beats scaling.
p_fresh is held constant (KDE-based) across all runs to isolate lambda estimation quality.

Four estimators:
- **A: Naive rolling** — reviews in last 24h / 24
- **B: Blended rolling** — prior rate blended with rolling average
- **C: KDE (scaled)** — existing per-critic KDE model with multiplicative scaling
- **D: Blended KDE** — unscaled KDE rate blended with rolling rate (like p_fresh blend)

See `plans/plan_lambda_baseline_comparison.md` and `plans/plan_blended_kde_lambda.md`.

In [ ]:
import sys, os, io, contextlib, warnings, time
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from edge import compute_edge
from critic_model import (
    build_critic_profiles,
    build_kde_lambda_model,
    default_training_slugs,
    estimate_lambda,
    estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)

movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])

movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")

print(f"Reviews: {len(reviews_df):,} rows, {reviews_df['movie_slug'].nunique()} movies")
print(f"Movies for backtest: {len(movies_bt)}")

## Helper functions

Copied from `kde_backtest.ipynb` — shared infrastructure.

In [ ]:
def load_hourly_prices(slug):
    """Load and forward-fill the hourly price CSV for a movie."""
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files:
        return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    df[thresh_cols] = df[thresh_cols].ffill()
    return df


def get_resolution(price_df):
    """Derive resolution from terminal prices. Returns {threshold: bool or None}."""
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    resolution = {}
    for col in thresh_cols:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty:
            resolution[thresh] = None
            continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90:
            resolution[thresh] = True
        elif terminal <= 10:
            resolution[thresh] = False
        else:
            resolution[thresh] = None
    return resolution


def precompute_review_states(slug, reviews_df, bet_close):
    """Precompute cumulative review states sorted by timestamp."""
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    if movie_reviews.empty:
        return [], None
    states = []
    critics = set()
    fresh = 0
    total = 0
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive":
            fresh += 1
        states.append({
            "timestamp": row["estimated_timestamp"],
            "observed_critics": frozenset(critics),
            "fresh_count": fresh,
            "total_count": total,
        })
    return states, movie_reviews["estimated_timestamp"].iloc[0]


def get_review_state_at(states, snapshot_time):
    """Binary search for review state at snapshot_time."""
    if not states or snapshot_time < states[0]["timestamp"]:
        return set(), 0, 0
    lo, hi = 0, len(states) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if states[mid]["timestamp"] <= snapshot_time:
            lo = mid
        else:
            hi = mid - 1
    s = states[lo]
    return set(s["observed_critics"]), s["fresh_count"], s["total_count"]

## Lambda estimator functions

In [ ]:
def naive_rolling_lambda(review_states, snapshot_time, hours_to_close, lookback_hours=24):
    """Estimator A: reviews in last 24h / 24. Returns reviews/hour."""
    if hours_to_close <= 0:
        return 0.0
    lookback_time = snapshot_time - timedelta(hours=lookback_hours)
    _, _, total_now = get_review_state_at(review_states, snapshot_time)
    _, _, total_before = get_review_state_at(review_states, lookback_time)
    count_in_window = total_now - total_before
    return count_in_window / lookback_hours


def compute_prior_rate(reviews_df, movies_df, training_slugs):
    """Compute average reviews/hour across training movies.
    
    For each training movie: total reviews before close / review window length in hours.
    Returns the mean across training movies.
    """
    rates = []
    for slug in training_slugs:
        movie_reviews = reviews_df[reviews_df["movie_slug"] == slug]
        bet_close = movies_df[movies_df["Slug"] == slug]["Bet Close Date"].iloc[0]
        movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
        if len(movie_reviews) < 2:
            continue
        first_ts = movie_reviews["estimated_timestamp"].min()
        window_hours = (bet_close - first_ts).total_seconds() / 3600
        if window_hours <= 0:
            continue
        rates.append(len(movie_reviews) / window_hours)
    return np.mean(rates) if rates else 0.1  # fallback


def blended_rolling_lambda(review_states, snapshot_time, hours_to_close,
                           total_count, prior_rate, n_lambda_prior=20, lookback_hours=24):
    """Estimator B: prior-blended rolling average. Returns reviews/hour."""
    if hours_to_close <= 0:
        return 0.0
    observed_rate = naive_rolling_lambda(review_states, snapshot_time, hours_to_close, lookback_hours)
    w = total_count / (total_count + n_lambda_prior)
    return w * observed_rate + (1 - w) * prior_rate


def blended_kde_lambda(model, days_before_close, hours_to_close,
                       observed_critics, review_states, snapshot_time,
                       total_count, n_blend=20, lookback_hours=24):
    """Estimator D: unscaled KDE rate blended with rolling observed rate.
    
    Prior: KDE expected remaining / hours_to_close (no scaling step).
    Observed: rolling rate (reviews in last 24h / 24).
    Blend weight shifts toward observed as total_count grows.
    """
    if hours_to_close <= 0:
        return 0.0
    kde_rate = estimate_lambda(
        model, days_before_close, hours_to_close, observed_critics,
    )
    rolling_rate = naive_rolling_lambda(
        review_states, snapshot_time, hours_to_close, lookback_hours,
    )
    w = total_count / (total_count + n_blend)
    return w * rolling_rate + (1 - w) * kde_rate

## Backtest loop

Single pass per movie, three lambda estimates per snapshot. Shared p_fresh (KDE-based).

In [ ]:
def backtest_movie_comparison(slug, reviews_df, movies_df, every_n_hours=24):
    """Run backtest with all four lambda estimators. Returns list of trade records."""
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]

    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty:
        return []

    market_close_time = price_df["timestamp"].iloc[-1]
    resolution = get_resolution(price_df)

    training_slugs = default_training_slugs(
        movies_df, exclude_slug=slug, before_date=bet_close_date
    )
    if len(training_slugs) < 5:
        return []

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)

    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    prior_rate = compute_prior_rate(reviews_df, movies_df, training_slugs)

    # Final total for lambda accuracy
    _, _, final_total = get_review_state_at(review_states, market_close_time)

    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    records = []

    prev_total = -1
    last_kept_ts = None

    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]

        if every_n_hours > 1 and last_kept_ts is not None:
            hours_since = (snapshot_time - last_kept_ts).total_seconds() / 3600
            if hours_since < every_n_hours:
                continue
        last_kept_ts = snapshot_time

        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        if hours_to_close <= 0:
            continue

        days_before_close = hours_to_close / 24

        observed_critics, fresh_count, total_count = get_review_state_at(
            review_states, snapshot_time
        )

        state_changed = (total_count != prev_total)
        prev_total = total_count

        # p_fresh — shared across all estimators, only recompute on state change
        if state_changed or 'p_fresh' not in dir():
            p_fresh = estimate_p_fresh(
                profiles, observed_critics, fresh_count, total_count,
            )

        # Compute all four lambdas
        first_review_dbc = None
        if first_review_ts is not None and total_count > 0:
            first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400

        lambdas = {
            "kde": estimate_lambda(
                model, days_before_close, hours_to_close,
                observed_critics, observed_count=total_count,
                first_review_dbc=first_review_dbc,
            ),
            "naive_rolling": naive_rolling_lambda(
                review_states, snapshot_time, hours_to_close,
            ),
            "blended_rolling": blended_rolling_lambda(
                review_states, snapshot_time, hours_to_close,
                total_count, prior_rate,
            ),
            "blended_kde": blended_kde_lambda(
                model, days_before_close, hours_to_close,
                observed_critics, review_states, snapshot_time,
                total_count,
            ),
        }

        actual_remaining = final_total - total_count

        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            if pd.isna(market_price):
                continue
            resolved = resolution.get(thresh)
            if resolved is None:
                continue

            for est_name, lam in lambdas.items():
                try:
                    result = compute_edge(
                        threshold=thresh,
                        market_price=market_price,
                        fresh_count=fresh_count,
                        total_count=total_count,
                        hours_to_close=hours_to_close,
                        lambda_rate=lam,
                        p_fresh=p_fresh,
                    )
                except (ValueError, Exception):
                    continue

                records.append({
                    "slug": slug,
                    "estimator": est_name,
                    "snapshot_time": snapshot_time,
                    "hours_to_close": hours_to_close,
                    "threshold": thresh,
                    "market_price": market_price,
                    "model_p_yes": result["p_yes"],
                    "edge_cents": result["edge_cents"],
                    "resolved_yes": resolved,
                    "lambda_rate": lam,
                    "p_fresh": p_fresh,
                    "fresh_count": fresh_count,
                    "total_count": total_count,
                    "expected_reviews": result["expected_reviews"],
                    "actual_remaining": actual_remaining,
                })

    return records

In [ ]:
EVERY_N_HOURS = 24

all_records = []
slugs = movies_bt["Slug"].tolist()
skipped = []
t0 = time.time()

for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)", end="", flush=True)
    try:
        records = backtest_movie_comparison(slug, reviews_df, movies_df, every_n_hours=EVERY_N_HOURS)
        all_records.extend(records)
    except Exception as e:
        skipped.append((slug, str(e)))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.0f}s. {len(all_records):,} evaluations across {len(slugs) - len(skipped)} movies.")
if skipped:
    print(f"Skipped {len(skipped)}:")
    for s, err in skipped:
        print(f"  {s}: {err}")

In [ ]:
trades = pd.DataFrame(all_records)
trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")
trades["abs_edge"] = trades["edge_cents"].abs()

# P&L per contract
trades["pnl_yes"] = np.where(trades["resolved_yes"], 100 - trades["market_price"], -trades["market_price"])
trades["pnl_no"] = np.where(~trades["resolved_yes"], trades["market_price"], -(100 - trades["market_price"]))

# Lambda prediction error
trades["predicted_remaining"] = trades["expected_reviews"]
trades["lambda_error"] = trades["predicted_remaining"] - trades["actual_remaining"]

print(f"Shape: {trades.shape}")
print(f"Movies: {trades['slug'].nunique()}")
print(f"\nRecords per estimator:")
print(trades.groupby("estimator").size())

# Sanity: all four estimators have same snapshot count
counts = trades.groupby("estimator").size()
assert counts.nunique() == 1, f"Estimator record counts differ: {counts.to_dict()}"

## 1. P&L comparison (No-only)

In [ ]:
ACTION_WINDOW = (24, 120)  # T-5d to T-1d
ESTIMATORS = ["kde", "blended_kde", "naive_rolling", "blended_rolling"]

rows = []
for est in ESTIMATORS:
    for me in [5, 10, 15, 20]:
        mask = (
            (trades["estimator"] == est) &
            (trades["direction"] == "No") &
            (trades["abs_edge"] >= me) &
            (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
            (trades["hours_to_close"] <= ACTION_WINDOW[1])
        )
        t = trades[mask]
        if len(t) == 0:
            continue
        wins = (t["pnl_no"] > 0).sum()
        rows.append({
            "estimator": est,
            "min_edge": me,
            "trades": len(t),
            "win_rate": f"{wins / len(t):.1%}",
            "total_pnl": f"{t['pnl_no'].sum():,.0f}",
            "mean_pnl": f"{t['pnl_no'].mean():.1f}",
            "movies": t["slug"].nunique(),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 2. Lambda accuracy: predicted vs actual remaining reviews

In [ ]:
# Filter to action window, one row per (slug, snapshot_time, estimator)
action = trades[
    (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
    (trades["hours_to_close"] <= ACTION_WINDOW[1])
].copy()

# Deduplicate: lambda is same across thresholds for a given (slug, snapshot, estimator)
snap_level = action.groupby(["estimator", "slug", "snapshot_time"]).first().reset_index()

print("Lambda prediction error (predicted_remaining - actual_remaining):")
print("  Negative = underprediction, Positive = overprediction\n")

for est in ESTIMATORS:
    sub = snap_level[snap_level["estimator"] == est]
    err = sub["lambda_error"]
    print(f"{est}:")
    print(f"  MAE:        {err.abs().mean():.1f} reviews")
    print(f"  Mean error: {err.mean():+.1f} reviews (bias)")
    print(f"  Median err: {err.median():+.1f} reviews")
    print()

## 3. Zero-lambda frequency (Estimator A)

In [ ]:
naive_action = snap_level[snap_level["estimator"] == "naive_rolling"]
n_zero = (naive_action["lambda_rate"] == 0).sum()
n_total = len(naive_action)
print(f"Naive rolling: {n_zero} / {n_total} snapshots have lambda=0 ({n_zero/n_total:.1%})")
print(f"  (These produce overconfident signals — model treats current score as final)")

# How many unique movies have at least one zero-lambda snapshot in action window?
zero_movies = naive_action[naive_action["lambda_rate"] == 0]["slug"].nunique()
total_movies = naive_action["slug"].nunique()
print(f"  {zero_movies} / {total_movies} movies affected")

## 4. P&L by time horizon

In [ ]:
MIN_EDGE = 10  # Use 10c for horizon breakdown

horizon_bins = [(24, 48, "T-1d to T-2d"), (48, 72, "T-2d to T-3d"),
                (72, 96, "T-3d to T-4d"), (96, 120, "T-4d to T-5d")]

no_trades = trades[
    (trades["direction"] == "No") &
    (trades["abs_edge"] >= MIN_EDGE) &
    (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
    (trades["hours_to_close"] <= ACTION_WINDOW[1])
].copy()

rows = []
for est in ESTIMATORS:
    for lo, hi, label in horizon_bins:
        mask = (
            (no_trades["estimator"] == est) &
            (no_trades["hours_to_close"] >= lo) &
            (no_trades["hours_to_close"] < hi)
        )
        t = no_trades[mask]
        if len(t) == 0:
            continue
        rows.append({
            "estimator": est,
            "horizon": label,
            "trades": len(t),
            "total_pnl": t["pnl_no"].sum(),
            "mean_pnl": t["pnl_no"].mean(),
        })

horizon_df = pd.DataFrame(rows)
pivot = horizon_df.pivot(index="horizon", columns="estimator", values="mean_pnl")
pivot = pivot[ESTIMATORS]
print("Mean P&L per trade (cents) by horizon, No-only, min_edge=10c:")
print(pivot.to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean P&L per trade (cents)")
ax.set_title(f"No-only mean P&L by horizon (min_edge={MIN_EDGE}c)")
ax.axhline(0, color="k", linewidth=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Per-movie scatter: KDE P&L vs baselines

In [ ]:
MIN_EDGE_SCATTER = 10

no_action = trades[
    (trades["direction"] == "No") &
    (trades["abs_edge"] >= MIN_EDGE_SCATTER) &
    (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
    (trades["hours_to_close"] <= ACTION_WINDOW[1])
].copy()

movie_pnl = no_action.groupby(["estimator", "slug"])["pnl_no"].sum().unstack(level=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

comparisons = [
    ("naive_rolling", "KDE vs Naive"),
    ("blended_rolling", "KDE vs Blended Rolling"),
    ("blended_kde", "KDE (scaled) vs Blended KDE"),
]

for ax, (baseline, title) in zip(axes, comparisons):
    both = movie_pnl.dropna(subset=["kde", baseline])
    ax.scatter(both[baseline], both["kde"], s=15, alpha=0.6)
    lims = [min(both[[baseline, "kde"]].min().min(), -100),
            max(both[[baseline, "kde"]].max().max(), 100)]
    ax.plot(lims, lims, "k--", linewidth=0.5, label="y=x")
    ax.set_xlabel(f"{baseline} P&L (cents)")
    ax.set_ylabel("KDE (scaled) P&L (cents)")
    ax.set_title(title)
    ax.legend()

    kde_wins = (both["kde"] > both[baseline]).sum()
    total = len(both)
    ax.text(0.05, 0.95, f"KDE wins: {kde_wins}/{total} movies",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()

## Findings summary

In [ ]:
# Print a compact comparison
print("=" * 70)
print("LAMBDA BASELINE COMPARISON — SUMMARY (4 estimators)")
print("=" * 70)
print()
print("P&L (No-only, T-1d to T-5d, min_edge=10c):")
for est in ESTIMATORS:
    mask = (
        (trades["estimator"] == est) &
        (trades["direction"] == "No") &
        (trades["abs_edge"] >= 10) &
        (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades["hours_to_close"] <= ACTION_WINDOW[1])
    )
    t = trades[mask]
    wins = (t["pnl_no"] > 0).sum()
    print(f"  {est:<20s}: {t['pnl_no'].sum():>8,.0f}c total, "
          f"{t['pnl_no'].mean():>+5.1f}c/trade, "
          f"{wins/len(t):.0%} win rate, {len(t)} trades")

print()
print("Lambda prediction bias (mean error, action window):")
for est in ESTIMATORS:
    sub = snap_level[snap_level["estimator"] == est]
    print(f"  {est:<20s}: {sub['lambda_error'].mean():+.1f} reviews")

print()
naive_z = snap_level[(snap_level["estimator"] == "naive_rolling") & (snap_level["lambda_rate"] == 0)]
print(f"Naive zero-lambda snapshots: {len(naive_z)} / {len(snap_level[snap_level['estimator'] == 'naive_rolling'])}")